In [1]:
from __future__ import annotations
import re
import numpy as np
import pandas as pd

import config
from config import DatasetConfig, ADNIMERGE

In [2]:
# --- funzioni atomiche: ognuna prende un DataFrame e ne restituisce uno nuovo -
def load(cfg):
    return pd.read_csv(cfg.source, low_memory=False)

Sostituisce i valori "sentinella" — cioè codici usati per rappresentare dati mancanti in modo mascherato (non un vero NaN, ma un valore convenzionale come -4, "Unknown", "N/A", 999, ecc.) — con veri valori mancanti (NaN), che pandas può gestire correttamente.

In [3]:
def replace_unknown(df):
    return df.replace(config.UNKNOWN_SENTINELS, np.nan)

"ripulire" colonne numeriche che contengono simboli di censura

In [4]:
def decensor(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            s = df[c].astype(str).str.replace(r"^[<>]", "", regex=True)
            df[c] = pd.to_numeric(s, errors="coerce")
    return df

Elimina le righe dove tutte le colonne indicate sono vuote contemporaneamente

In [5]:
def drop_if_all_none(df, cols):
    present = [c for c in cols if c in df.columns]
    return df.dropna(subset=present, how="all") if present else df

Converte una o più colonne in formato data vera e propria (datetime), invece di lasciarle come semplice testo.

In [6]:
def parse_dates(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

Elimina le visite duplicate (stesso paziente, stessa data) tenendo però la riga più completa tra i duplicati, invece di sceglierla a caso.

In [7]:
def dedup_visits(df, id_col, date_col, essential):
    present = [c for c in essential if c in df.columns]
    df = df.copy()
    df["_completeness"] = df[present].notna().sum(axis=1) if present else 0
    return (df.sort_values([id_col, date_col, "_completeness"])
              .drop_duplicates(subset=[id_col, date_col], keep="last")
              .drop(columns="_completeness"))

Calcola, per ogni visita, a quanti mesi di distanza è avvenuta rispetto alla prima visita di quel paziente (il cosiddetto "baseline").

In [8]:
def add_visit_month(df, id_col, date_col):
    df = df.copy().sort_values([id_col, date_col])
    baseline = df.groupby(id_col)[date_col].transform("min")
    df["VISIT_MONTH"] = ((df[date_col] - baseline).dt.days / 30.44).round().astype("Int64")
    return df

Ricalcola l'età del paziente visita per visita, partendo dall'età al basale (prima visita) e sommando il tempo trascorso — invece di usare un'unica età fissa (quella al baseline) ripetuta erroneamente su tutte le visite successive.

In [9]:
def recompute_age(df, date_col="EXAMDATE", bl_date_col="EXAMDATE_bl", age_col="AGE"):
    if age_col not in df.columns or bl_date_col not in df.columns:
        return df
    df = df.copy().rename(columns={age_col: age_col + "_bl"})
    delta_y = (pd.to_datetime(df[date_col], errors="coerce")
               - pd.to_datetime(df[bl_date_col], errors="coerce")).dt.days / 365.25
    df[age_col] = df[age_col + "_bl"] + delta_y
    return df

Sostituisce i valori di determinate colonne con nuovi valori equivalenti, seguendo una mappatura predefinita salvata altrove (in un file/modulo config).

In [10]:
def recode(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in config.RECODE:
            df[c] = df[c].map(config.RECODE[c])
    return df

Ripulisce due colonne specifiche legate ai dati di risonanza magnetica processati con FreeSurfer (software comune in neuroimaging per l'analisi di risonanze cerebrali): FLDSTRENG (intensità del campo magnetico dello scanner MRI) e FSVERSION (versione del software FreeSurfer usata).

In [11]:
def clean_fs_fields(df):
    df = df.copy()
    if "FLDSTRENG" in df.columns:
        df["FLDSTRENG"] = df["FLDSTRENG"].astype(str).str.extract(r"([0-9.]+)", expand=False) + "T"
    if "FSVERSION" in df.columns:
        df["FSVERSION"] = df["FSVERSION"].astype(str).str.extract(r"([0-9.]+)", expand=False)
    return df

Rinomina le colonne del DataFrame usando una mappa di corrispondenze (vecchio nome → nuovo nome) definita centralmente in config.py.

In [12]:
def rename_variables(df):
    return df.rename(columns=config.rename_map())


Crea nuove colonne calcolando dei rapporti (ratio) tra biomarcatori del liquido cerebrospinale (CSF, Cerebrospinal Fluid) — misure molto usate negli studi sull'Alzheimer come ADNI.

In [13]:
def add_ratios(df):
    df = df.copy()
    for name, (num, den) in {"TTAU_AB42_CSF": ("TTAU_CSF", "AB42_CSF"),
                             "PT181_AB42_CSF": ("PT181_CSF", "AB42_CSF")}.items():
        if num in df.columns and den in df.columns:
            df[name] = df[num] / df[den]
    return df

Calcola il profilo ATN del paziente — un framework clinico standard usato negli studi sull'Alzheimer per classificare i pazienti in base alla presenza di tre tipi di biomarcatori: Amiloide, Tau, Neurodegenerazione.

In [14]:
def add_atn_profile(df, method):
    df = df.copy()
    axes = {"A": ("AB42_CSF", "below"), "T": ("PT181_CSF", "above"), "N": ("TTAU_CSF", "above")}
    flags = {}
    for axis, (var, direction) in axes.items():
        thr = config.cutoff(var, method)
        if var in df.columns and thr is not None:
            pos = df[var] < thr if direction == "below" else df[var] > thr
            flags[axis] = np.where(df[var].isna(), np.nan, pos.astype(float))
    if not flags:
        raise RuntimeError(f"compute_atn=True ma nessun asse calcolabile (metodo '{method}').")
    for axis, name in [("A", "Apositive"), ("T", "Tpositive"), ("N", "Npositive")]:
        if axis in flags:
            df[name] = flags[axis]
    df["ATN_PROFILE"] = df.apply(
        lambda r: "".join(f"{a}{'+' if r.get(n) == 1 else '-'}"
                          for a, n in [("A", "Apositive"), ("T", "Tpositive"), ("N", "Npositive")]
                          if n in df.columns and not pd.isna(r.get(n))) or np.nan, axis=1)
    return df

--- orchestratore: l'ex "adni_cleaning1" per un file, leggibile in un colpo --
E' l'orchestratore principale della pipeline di data cleaning: chiama in sequenza tutte le funzioni che abbiamo analizzato finora, componendole in un unico flusso completo e configurabile.

In [15]:
def run_cleaning1(cfg: DatasetConfig = ADNIMERGE) -> pd.DataFrame:
    df = load(cfg)                                                # download -> CSV
    df = replace_unknown(df)                                      # replace_unknown_values
    if cfg.decensor_biomarkers:
        df = decensor(df, config.columns_in("Biomarker"))
    df = drop_if_all_none(df, cfg.essential_columns)             # 1° drop: colonne importanti
    df = drop_if_all_none(df, cfg.also_required)                 # 2° drop: DX obbligatoria
    df = parse_dates(df, [cfg.date_column])                      # to_date_format
    df = dedup_visits(df, cfg.id_column, cfg.date_column, cfg.essential_columns)
    df = add_visit_month(df, cfg.id_column, cfg.date_column)     # find_exam_code -> VISIT_MONTH
    if cfg.recompute_age:
        df = recompute_age(df, cfg.date_column)                  # add_calculated_age
    df = recode(df, cfg.recode_columns)                          # categorize_*
    if cfg.clean_fs_fields:
        df = clean_fs_fields(df)                                 # FLDSTRENG/FSVERSION
    df = rename_variables(df)                                     # new_variable_names
    if cfg.compute_atn:                                          # solo file CSF
        df = add_ratios(df)
        df = add_atn_profile(df, cfg.atn_method)
    return df

--- report: rigenera cio' che prima era l'Excel _statistics (output, non input)
Genera un report riepilogativo (profilazione) di tutte le colonne del dataset, riga per colonna, includendo anche un'analisi per coorte di studio — molto utile come passaggio finale dopo la pipeline di cleaning, per avere una visione d'insieme della qualità dei dati.

In [16]:
def profile(df, cohort_col="COLPROT") -> pd.DataFrame:
    rows, n = [], len(df)
    cohorts = df[cohort_col].dropna().unique().tolist() if cohort_col in df else []
    for col in df.columns:
        s = df[col]
        n_valid = int(s.notna().sum())
        is_num = pd.api.types.is_numeric_dtype(s)
        rows.append({
            "variable": col,
            "type": s.dtype.name,
            "range": f"{s.min()}, {s.max()}" if is_num and n_valid else "",
            "valid_values": n_valid,
            "missing_values": n - n_valid,
            "missing_pop": ", ".join(c for c in cohorts
                                     if df.loc[df[cohort_col] == c, col].notna().sum() == 0),
            "del": "keep" if n and n_valid / n >= config.MISSING_KEEP_THRESHOLD else "drop",
        })
    return pd.DataFrame(rows)

Il blocco di esecuzione principale dello script: il codice che viene effettivamente eseguito quando lanci il file .py direttamente (non quando viene importato come modulo in un altro script).

In [18]:
if __name__ == "__main__":
    df = run_cleaning1()
    print(f"cleaning1 ADNIMERGE: {df.shape[0]} righe x {df.shape[1]} colonne")
    df.to_csv("ADNIMERGE_cleaned_01.csv", index=False)           # ex upload level='cleaned_01'
    profile(df).to_csv("ADNIMERGE_report.csv", index=False)      # ex support file statistics
    print("scritti: ADNIMERGE_cleaned_01.csv, ADNIMERGE_report.csv")

cleaning1 ADNIMERGE: 11458 righe x 118 colonne
scritti: ADNIMERGE_cleaned_01.csv, ADNIMERGE_report.csv


Prova visualizzazione prime 15 righe

In [19]:
df.head(15)

,RID,COLPROT,ORIGPROT,PTID,SITE,VISCODE,EXAMDATE,DX_bl,AGE_bl,GENDER,...,PIB_bl,AV45_bl,FBB_bl,Years_bl,Month_bl,Month,M,update_stamp,VISIT_MONTH,AGE
0,2,ADNI1,ADNI1,011_S_0002,11,bl,2005-09-08,CN,74.3,1,...,NaN,NaN,NaN,0.000000,0.00000,0,0,2023-07-07 04:59:40,0,74.300000
5105,2,ADNI1,ADNI1,011_S_0002,11,m06,2006-03-06,CN,74.3,1,...,NaN,NaN,NaN,0.490075,5.86885,6,6,2023-07-07 04:59:40,6,74.790075
5106,2,ADNI1,ADNI1,011_S_0002,11,m36,2008-08-27,CN,74.3,1,...,NaN,NaN,NaN,2.967830,35.54100,36,36,2023-07-07 04:59:40,36,77.267830
5107,2,ADNIGO,ADNI1,011_S_0002,11,m60,2010-09-22,CN,74.3,1,...,NaN,NaN,NaN,5.037650,60.32790,60,60,2023-07-07 04:59:40,60,79.337645
11172,2,ADNI2,ADNI1,011_S_0002,11,m72,2011-09-19,CN,74.3,1,...,NaN,NaN,NaN,6.028750,72.19670,72,72,2023-07-07 04:59:40,72,80.328747
11170,2,ADNI2,ADNI1,011_S_0002,11,m84,2012-09-26,CN,74.3,1,...,NaN,NaN,NaN,7.049970,84.42620,84,84,2023-07-07 04:59:40,85,81.349966
11168,2,ADNI2,ADNI1,011_S_0002,11,m96,2013-09-09,CN,74.3,1,...,NaN,NaN,NaN,8.002740,95.83610,96,96,2023-07-07 04:59:40,96,82.302738
11165,2,ADNI2,ADNI1,011_S_0002,11,m120,2015-09-22,CN,74.3,1,...,NaN,NaN,NaN,10.037000,120.19700,120,120,2023-07-07 04:59:40,120,84.336961
11233,2,ADNI2,ADNI1,011_S_0002,11,m132,2016-09-27,CN,74.3,1,...,NaN,NaN,NaN,11.052700,132.36100,132,132,2023-07-07 04:59:40,133,85.352704
11234,2,ADNI3,ADNI1,011_S_0002,11,m144,2017-10-18,CN,74.3,1,...,NaN,NaN,NaN,12.109500,145.01600,144,144,2023-07-07 04:59:40,145,86.409514
